# 🛰️ Nagpur Lakes — Real Sentinel-1 × Sentinel-2 Debris Overlay

Runs the cross-modal (RANSAC + CPD) floating-debris pipeline on **real, live** satellite data:

* **Sentinel-2 (optical)** — most recent *clear* scene over a chosen Nagpur lake (public AWS bucket).
* **Sentinel-1 (SAR)** — latest scene from **Microsoft Planetary Computer** (anonymous, no account).

Because the two sensors image at different times, floating debris **drifts** between passes — the
RANSAC+CPD step estimates that drift and lines the two debris clouds up so debris seen by **both**
sensors can be matched.

**How to run:** `Runtime → Run all` (or press ▶ on each cell). No credentials, no setup. ~2–3 min.

> Colab runs in Google's cloud, where the Sentinel-1/2 data hosts are reachable — so this produces the
> **fully-real** S1×S2 result.

## 1. Install dependencies & get the code

In [ ]:
import os
REPO = 'https://github.com/DevDhapodkar/PBL-Sem-7.git'
BRANCH = 'claude/sentinel-debris-detection-m17syi'
if not os.path.exists('PBL-Sem-7'):
    !git clone -q --branch $BRANCH $REPO
else:
    !cd PBL-Sem-7 && git pull -q    # get latest fixes on re-run
%cd PBL-Sem-7
# ensure no global GDAL extension restriction (S1 assets are .rtc.tiff)
os.environ.pop('CPL_VSIL_CURL_ALLOWED_EXTENSIONS', None)
print('ready')

## 2. Pick a lake

Options: `ambazari`, `futala`, `gorewada`, `gandhisagar`, `sonegaon`, `telangkhedi`, `sakkardara`, `naik`.

In [ ]:
LAKE = 'ambazari'   #@param {type:'string'}

## 3. Fetch the most-recent clear Sentinel-2 (real) and detect floating matter

In [ ]:
import matplotlib.pyplot as plt
from src.nagpur import get_lake
from src.acquire import fetch_latest_clear_s2
from src.detect_real import detect_optical, water_mask
from src import visualize_nagpur as vn

lake = get_lake(LAKE)
bs = fetch_latest_clear_s2(lake)
print('Sentinel-2:', bs.meta['scene'], bs.meta['date'], '(most recent clear)')

opt_dets, diag = detect_optical(bs)
print('optical floating-matter detections:', len(opt_dets))
from src.data_simulation import Scene
import numpy as np
scene_preview = Scene(np.empty((0,2)), [], opt_dets, {})
fig = vn.plot_scene_overview(bs, scene_preview, diag); plt.show()

## 4. Fetch the latest Sentinel-1 SAR (real, Planetary Computer) and detect debris

In [ ]:
from src.acquire import fetch_latest_s1_pc
from src.detect_real import detect_sar_backscatter

bs_sar = fetch_latest_s1_pc(lake, bs)
print('Sentinel-1:', bs_sar.meta['scene'], bs_sar.meta['date'], '(latest RTC)')
sar_dets = detect_sar_backscatter(bs_sar, diag['water'])
print('SAR debris detections:', len(sar_dets))

## 5. Register (RANSAC → CPD) and overlay — drift-aware comparison

In [ ]:
import datetime as dt, numpy as np
from src.data_simulation import Scene
from src.pipeline import run_proposed

scene = Scene(np.empty((0,2)), sar_dets, opt_dets, {'matrix_sar_to_opt': np.eye(3)})
proposed = run_proposed(scene, putative_radius=16, ransac_threshold=5, match_radius=6)

drift_px = float(np.hypot(*proposed.T_final[:2,2]))
print(f'estimated debris drift / offset: ~{drift_px:.1f}px (~{drift_px*10:.0f} m)')
print(f'D_reg (RMSE): {proposed.fusion.d_registration:.2f}px')
print(f'debris corroborated by BOTH sensors: {proposed.fusion.n_matches}')
try:
    gap = abs((dt.date.fromisoformat(bs.meta['date']) - dt.date.fromisoformat(bs_sar.meta['date'])).days)
    print(f'S1-S2 acquisition gap: {gap} days')
except Exception: pass

fig = vn.plot_overlay(bs, scene, proposed, s1_date=bs_sar.meta['date'], s2_date=bs.meta['date'])
plt.show()

**Left** — raw overlay: the S1 (▲) and S2 (○) debris are offset by the inter-pass drift + co-registration.
**Right** — after RANSAC+CPD registration the SAR lines up with the optical, and debris seen by **both**
sensors is ringed in yellow.

---
### One-liner alternative
You can also just run the command-line script (same result):

In [ ]:
!python fetch_and_overlay.py --lake $LAKE --s1 pc --out results/overlay.png
from IPython.display import Image; Image('results/overlay.png')